In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import torch

In [ ]:

df = pd.read_csv(
    '../data/news_dataset_balanced.csv').dropna(subset=["title", "category"])

In [3]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["category"])

In [4]:
train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42)

In [5]:
from transformers import DistilBertTokenizer

In [6]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")


def tokenize_texts(texts, max_len=128):
    return tokenizer(
        texts.tolist(),
        padding='max_length',
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

In [7]:
train_encodings = tokenize_texts(train_df["title"])
test_encodings = tokenize_texts(test_df["title"])

train_labels = torch.tensor(train_df["label"].values)
test_labels = torch.tensor(test_df["label"].values)

In [8]:
from torch.utils.data import Dataset, DataLoader

In [21]:
class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


train_dataset = NewsDataset(train_encodings, train_labels)
test_dataset = NewsDataset(test_encodings, test_labels)

train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=10, shuffle=False)

In [ ]:
import os
os.environ["TRANSFORMERS_NO_TORCHVISION_IMPORT"] = "1"

In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [11]:
num_labels = len(df["label"].unique())

In [12]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

/home/pranydotin/Projects/update/venv/lib/python3.12/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'pre_classifier.weight', 'classifier.weight', 'pre_classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
 

In [14]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [15]:
from tqdm import tqdm

(503, 3)

In [27]:
epochs = 5

for epoch in range(epochs):
    model.train()
    loop = tqdm(train_loader, leave=True)
    total_loss = 0
    for batch in loop:
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(
            input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        # Backward pass
        loss.backward()
        optimizer.step()

        # Update progress bar
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())
    print(f"Average loss: {total_loss/len(train_loader):.4f}")

Epoch 1: 100%|██████████| 41/41 [01:45<00:00,  2.57s/it, loss=1.21]


Average loss: 1.9005


Epoch 2: 100%|██████████| 41/41 [01:42<00:00,  2.50s/it, loss=1.26] 


Average loss: 1.2752


Epoch 3: 100%|██████████| 41/41 [01:42<00:00,  2.51s/it, loss=0.298]


Average loss: 0.6752


Epoch 4: 100%|██████████| 41/41 [01:44<00:00,  2.54s/it, loss=0.147]


Average loss: 0.3492


Epoch 5: 100%|██████████| 41/41 [02:03<00:00,  3.01s/it, loss=0.157]

Average loss: 0.1966


In [29]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [30]:
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [31]:
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='weighted')
recall = recall_score(all_labels, all_preds, average='weighted')
f1 = f1_score(all_labels, all_preds, average='weighted')

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

Accuracy:  0.8911
Precision: 0.8924
Recall:    0.8911
F1 Score:  0.8868


In [ ]:
save_directory = "./distilbert_news_model"

model.save_pretrained(save_directory)

tokenizer.save_pretrained(save_directory)

('./distilbert_news_model/tokenizer_config.json',
 './distilbert_news_model/special_tokens_map.json',
 './distilbert_news_model/vocab.txt',
 './distilbert_news_model/added_tokens.json',
 './distilbert_news_model/tokenizer.json')